In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.llms import llm_factory
from ragas.metrics._context_precision import context_precision
from ragas.metrics._context_recall import context_recall
from ragas.metrics._faithfulness import faithfulness

ImportError: cannot import name 'Mistral' from 'mistralai' (unknown location)

In [ ]:
load_dotenv()

# Ewaluacja RAGa

Jednym z narzędzi do ewaluacji systemów RAG jest biblioteka `ragas`

https://docs.ragas.io/en/latest/getstarted/

## Quickstart

`ragas` dostarcza trzy kluczowe obiekty do ewaluacji systemów RAG:

- `SingleTurnSample` – pojedynczy przypadek testowy: pytanie użytkownika (`user_input`), pobrane konteksty (`retrieved_contexts`), odpowiedź modelu (`response`) oraz opcjonalnie wzorcowa odpowiedź (`reference`)
- `EvaluationDataset` – kolekcja przypadków testowych przekazywana do ewaluacji
- `evaluate()` – synchroniczna funkcja obliczająca wybrane metryki dla całego datasetu; zwraca wyniki zagregowane oraz per-sample przez `results.to_pandas()`

Metryki w `ragas` wymagają modelu LLM — przekazujemy go do `evaluate()`, nie do konstruktora metryki.

In [ ]:
openai_client = OpenAI()
eval_llm = llm_factory("gpt-4o-mini", client=openai_client)

Dokumentacja metryk: https://docs.ragas.io/en/latest/concepts/metrics/overview/

In [ ]:
user_input = "Jaka jest stolica Francji?"
response = "Stolicą Francji jest Paryż."
contexts = [
    "Paryż jest stolicą i największym miastem Francji.",
    "Wieża Eiffla znajduje się w centrum Paryża i jest symbolem Francji.",
    "Berlin jest stolicą Niemiec."
]
reference = "Stolicą Francji jest Paryż."

sample = SingleTurnSample(
    user_input=user_input,
    retrieved_contexts=contexts,
    response=response,
    reference=reference
)

dataset = EvaluationDataset(samples=[sample])
results = evaluate(
    dataset=dataset,
    metrics=[context_precision, context_recall, faithfulness],
    llm=eval_llm
)
results

## Metryki

W bibliotece `ragas` dostępne są metryki odpowiadające tym z `deepeval`:

- `context_precision` (odpowiednik `ContextualPrecisionMetric`)
- `context_recall` (odpowiednik `ContextualRecallMetric`)
- `faithfulness` (odpowiednik `FaithfulnessMetric`)

Metryki tworzymy bez argumentów i przekazujemy do `evaluate()` razem z `llm`.

### `context_precision`

Ocenia, czy bardziej istotne fragmenty kontekstu znalazły się wyżej na liście pobranych dokumentów.
Czyli sprawdzamy, czy kolejność pozyskanych dokumentów jest optymalna względem oczekiwanej odpowiedzi.

https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_precision/

---

**Zestawiamy:** kolejność elementów w `retrieved_contexts` i `reference`

In [ ]:
# Dobra kolejność – relewantny kontekst na pierwszym miejscu
sample_1 = SingleTurnSample(
    user_input="Kiedy Polska odzyskała niepodległość?",
    retrieved_contexts=[
        "Polska odzyskała niepodległość 11 listopada 1918 roku po ponad 123 latach zaborów.",
        "Józef Piłsudski był kluczową postacią w odzyskaniu przez Polskę niepodległości.",
        "Warszawa jest stolicą Polski od XVI wieku."
    ],
    response="Polska odzyskała niepodległość 11 listopada 1918 roku.",
    reference="Polska odzyskała niepodległość 11 listopada 1918 roku po ponad 123 latach zaborów."
)

# Odwrócona kolejność – relewantny kontekst na końcu
sample_2 = SingleTurnSample(
    user_input="Kiedy Polska odzyskała niepodległość?",
    retrieved_contexts=[
        "Warszawa jest stolicą Polski od XVI wieku.",
        "Józef Piłsudski był kluczową postacią w odzyskaniu przez Polskę niepodległości.",
        "Polska odzyskała niepodległość 11 listopada 1918 roku po ponad 123 latach zaborów."
    ],
    response="Polska odzyskała niepodległość 11 listopada 1918 roku.",
    reference="Polska odzyskała niepodległość 11 listopada 1918 roku po ponad 123 latach zaborów."
)

# Brak relewantnego kontekstu – retriever nie zwrócił przydatnych informacji
sample_3 = SingleTurnSample(
    user_input="Kiedy Polska odzyskała niepodległość?",
    retrieved_contexts=[
        "Wisła jest najdłuższą rzeką w Polsce o długości 1047 km.",
        "Polska graniczy z siedmioma krajami, m.in. z Niemcami i Ukrainą."
    ],
    response="Nie wiem, kiedy Polska odzyskała niepodległość.",
    reference="Polska odzyskała niepodległość 11 listopada 1918 roku po ponad 123 latach zaborów."
)

dataset = EvaluationDataset(samples=[sample_1, sample_2, sample_3])
results = evaluate(dataset=dataset, metrics=[context_precision], llm=eval_llm)
results

### `context_recall`

Ocenia, czy retriever potrafi poprawnie dobrać właściwe informacje do tego, co jest w oczekiwanej odpowiedzi.

https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_recall/

---

**Zestawiamy:** `retrieved_contexts` oraz `reference`

In [ ]:
# Pełne pokrycie – kontekst zawiera wszystkie informacje z reference
sample_1 = SingleTurnSample(
    user_input="Czym jest Układ Słoneczny?",
    retrieved_contexts=[
        "Układ Słoneczny to układ planetarny, w centrum którego znajduje się Słońce.",
        "W skład Układu Słonecznego wchodzi 8 planet oraz inne ciała niebieskie, jak asteroidy i komety.",
        "Układ Słoneczny powstał około 4,6 miliarda lat temu z obłoku gazu i pyłu."
    ],
    response="Układ Słoneczny to układ planetarny ze Słońcem w centrum, zawierający 8 planet. Powstał 4,6 mld lat temu.",
    reference="Układ Słoneczny to układ planetarny ze Słońcem w centrum, zawierający 8 planet i inne ciała. Powstał 4,6 miliarda lat temu."
)

# Częściowe pokrycie – kontekst zawiera tylko część informacji z reference
sample_2 = SingleTurnSample(
    user_input="Czym jest Układ Słoneczny i ile ma planet?",
    retrieved_contexts=[
        "Układ Słoneczny to układ planetarny, w centrum którego znajduje się Słońce.",
        "Układ Słoneczny powstał około 4,6 miliarda lat temu z obłoku gazu i pyłu."
    ],
    response="Układ Słoneczny to układ planetarny ze Słońcem w centrum. Nie mam informacji o liczbie planet.",
    reference="Układ Słoneczny to układ planetarny ze Słońcem w centrum, zawierający 8 planet. Powstał 4,6 miliarda lat temu."
)

# Brak pokrycia – kontekst nie zawiera żadnych informacji z reference
sample_3 = SingleTurnSample(
    user_input="Czym jest Układ Słoneczny?",
    retrieved_contexts=[
        "Galaktyka Drogi Mlecznej zawiera od 200 do 400 miliardów gwiazd.",
        "Najjaśniejsza gwiazda na nocnym niebie to Syriusz w gwiazdozbiorze Wielkiego Psa."
    ],
    response="Nie mam informacji o Układzie Słonecznym w podanym kontekście.",
    reference="Układ Słoneczny to układ planetarny ze Słońcem w centrum, zawierający 8 planet."
)

dataset = EvaluationDataset(samples=[sample_1, sample_2, sample_3])
results = evaluate(dataset=dataset, metrics=[context_recall], llm=eval_llm)
results

### `faithfulness`

Określa, czy odpowiedź modelu odnosi się do pozyskanego z bazy kontekstu.
Sprawdza, czy wszystkie twierdzenia z odpowiedzi wynikają z kontekstu.

https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/faithfulness/

---

**Zestawiamy:** `response` i `retrieved_contexts`.

In [ ]:
# Odpowiedź w pełni wynika z kontekstu
sample_1 = SingleTurnSample(
    user_input="Czym jest Python?",
    retrieved_contexts=[
        "Python to interpretowany język programowania wysokiego poziomu, znany z czytelnej składni.",
        "Python jest szeroko stosowany w analizie danych, sztucznej inteligencji i tworzeniu aplikacji webowych."
    ],
    response="Python to interpretowany język programowania stosowany w analizie danych, AI i aplikacjach webowych."
)

# Odpowiedź częściowo wynika z kontekstu – zawiera twierdzenia spoza kontekstu (halucynacja)
sample_2 = SingleTurnSample(
    user_input="Czym jest Python?",
    retrieved_contexts=[
        "Python to interpretowany język programowania wysokiego poziomu, znany z czytelnej składni."
    ],
    response="Python to interpretowany język programowania stworzony przez Guido van Rossuma w 1991 roku, znany z czytelnej składni."
)

# Odpowiedź sprzeczna z kontekstem
sample_3 = SingleTurnSample(
    user_input="Czym jest Python?",
    retrieved_contexts=[
        "Python to język programowania wysokiego poziomu. Jest językiem interpretowanym, nie kompilowanym."
    ],
    response="Python to kompilowany język programowania niskiego poziomu, trudny w nauce."
)

dataset = EvaluationDataset(samples=[sample_1, sample_2, sample_3])
results = evaluate(dataset=dataset, metrics=[faithfulness], llm=eval_llm)
results

In [ ]:
results.to_pandas()

### Podsumowanie metryk

| Nazwa metryki (ragas)                | Odpowiednik w deepeval      | Zestawienie | Pola `SingleTurnSample` |
|--------------------------------------|-----------------------------|---|---|
| `context_precision`   | `ContextualPrecisionMetric` | kolejność `retrieved_contexts` i `reference` | `user_input, retrieved_contexts, reference` |
| `context_recall`                      | `ContextualRecallMetric`    | `retrieved_contexts` i `reference` | `user_input, retrieved_contexts, reference` |
| `faithfulness`                       | `FaithfulnessMetric`        | `response` i `retrieved_contexts` | `user_input, response, retrieved_contexts` |

## Ewaluacja RAGa – use case

Use case na przykładzie `knowledge_database`

In [ ]:
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai.chat_models import ChatOpenAI

In [ ]:
client = QdrantClient(url="http://localhost:6333")

In [ ]:
embeddings = HuggingFaceEmbeddings(model="sdadas/st-polish-paraphrase-from-distilroberta")

In [ ]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name="knowledge_database",
    embedding=embeddings,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
template = """Retrieved documents:
---------
{context}

-----
End of retrieved documents
============

Answer the following question: {input}.

=============

Important information: You can answer only based on the context from the retrieved documents.
If you don't have any information regarding the question in the context, say 'I don't know'"""


prompt_template = PromptTemplate.from_template(template)

In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt_template)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

---

In [ ]:
user_input = "Powiedz mi kto czym jeździ"
response_data = retrieval_chain.invoke({"input": user_input})

In [ ]:
reference = "Andrzej jeździ samochodem, Bartek rowerem a Czarek tramwajem."
retrieved_contexts = [doc.page_content for doc in response_data["context"]]

In [ ]:
retrieved_contexts

In [ ]:
response = response_data["answer"]
response

### Metryki

In [ ]:
dataset = EvaluationDataset(samples=[
    SingleTurnSample(
        user_input=user_input,
        retrieved_contexts=retrieved_contexts,
        response=response,
        reference=reference
    )
])

results = evaluate(dataset=dataset, metrics=[context_precision, context_recall, faithfulness], llm=eval_llm)
results